In [48]:
!pip install pyspark



In [91]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import desc
from pyspark.sql.functions import avg
from pyspark.sql.functions import countDistinct
from pyspark.sql.functions import col, sum, avg, countDistinct, desc


spark = SparkSession.builder.appName("DmartAnalysis").getOrCreate()

In [26]:
from google.colab import files
uploaded = files.upload()

Saving Customer.csv to Customer.csv
Saving Product.csv to Product.csv
Saving Sales.csv to Sales.csv


# Task 2: Load Data

In [50]:
Products_df = spark.read.csv("/content/Product.csv", header=True, inferSchema=True)
Sales_df    = spark.read.csv("/content/Sales.csv", header=True, inferSchema=True)
Customers_df= spark.read.csv("/content/Customer.csv", header=True, inferSchema=True)




In [51]:
# Check schema & first 2 rows
Products_df.printSchema()
Products_df.show(2, truncate=False)

root
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)

+---------------+---------+------------+-----------------------------------------------------------+
|Product ID     |Category |Sub-Category|Product Name                                               |
+---------------+---------+------------+-----------------------------------------------------------+
|FUR-BO-10001798|Furniture|Bookcases   |Bush Somerset Collection Bookcase                          |
|FUR-CH-10000454|Furniture|Chairs      |Hon Deluxe Fabric Upholstered Stacking Chairs  Rounded Back|
+---------------+---------+------------+-----------------------------------------------------------+
only showing top 2 rows



In [52]:
# Transformation - changing the column names
for col in Products_df.columns:
    if " " or "-" in col:
        Products_df = Products_df.withColumnRenamed(col, col.replace(" ", "_").replace("-","_"))

Products_df.show(2)

+---------------+---------+------------+--------------------+
|     Product_ID| Category|Sub_Category|        Product_Name|
+---------------+---------+------------+--------------------+
|FUR-BO-10001798|Furniture|   Bookcases|Bush Somerset Col...|
|FUR-CH-10000454|Furniture|      Chairs|Hon Deluxe Fabric...|
+---------------+---------+------------+--------------------+
only showing top 2 rows



In [53]:
for col in Sales_df.columns:
    if " " in col:
        Sales_df = Sales_df.withColumnRenamed(col, col.replace(" ", "_"))

Sales_df.show(2)

+----------+--------------+----------+----------+------------+-----------+---------------+------+--------+--------+-------+
|Order_Line|      Order_ID|Order_Date| Ship_Date|   Ship_Mode|Customer_ID|     Product_ID| Sales|Quantity|Discount| Profit|
+----------+--------------+----------+----------+------------+-----------+---------------+------+--------+--------+-------+
|         1|CA-2016-152156|2016-11-08|2016-11-11|Second Class|   CG-12520|FUR-BO-10001798|261.96|       2|     0.0|41.9136|
|         2|CA-2016-152156|2016-11-08|2016-11-11|Second Class|   CG-12520|FUR-CH-10000454|731.94|       3|     0.0|219.582|
+----------+--------------+----------+----------+------------+-----------+---------------+------+--------+--------+-------+
only showing top 2 rows



In [54]:
for col in Customers_df.columns:
    if " " in col:
        Customers_df = Customers_df.withColumnRenamed(col, col.replace(" ", "_"))

Customers_df.show(2)

+-----------+---------------+---------+---+-------------+-----------+----------+-----------+------+
|Customer_ID|  Customer_Name|  Segment|Age|      Country|       City|     State|Postal_Code|Region|
+-----------+---------------+---------+---+-------------+-----------+----------+-----------+------+
|   CG-12520|    Claire Gute| Consumer| 67|United States|  Henderson|  Kentucky|      42420| South|
|   DV-13045|Darrin Van Huff|Corporate| 31|United States|Los Angeles|California|      90036|  West|
+-----------+---------------+---------+---+-------------+-----------+----------+-----------+------+
only showing top 2 rows



In [39]:
# Handling the missing values
from pyspark.sql.functions import col, sum

missing_values_products = Products_df.select( [sum(col(column).isNull().cast("int")).alias(column) for column in Products_df.columns] )

missing_values_sales = Sales_df.select( [sum(col(column).isNull().cast("int")).alias(column) for column in Sales_df.columns] )

missing_values_customers = Customers_df.select( [sum(col(column).isNull().cast("int")).alias(column) for column in Customers_df.columns] )

In [40]:
missing_values_products.show()

+----------+--------+------------+------------+
|Product_ID|Category|Sub_Category|Product_Name|
+----------+--------+------------+------------+
|         0|       0|           0|           0|
+----------+--------+------------+------------+



In [41]:
missing_values_sales.show()

+----------+--------+----------+---------+---------+-----------+----------+-----+--------+--------+------+
|Order_Line|Order_ID|Order_Date|Ship_Date|Ship_Mode|Customer_ID|Product_ID|Sales|Quantity|Discount|Profit|
+----------+--------+----------+---------+---------+-----------+----------+-----+--------+--------+------+
|         0|       0|         0|        0|        0|          0|         0|    0|       0|       0|     0|
+----------+--------+----------+---------+---------+-----------+----------+-----+--------+--------+------+



In [42]:
missing_values_customers.show()

+-----------+-------------+-------+---+-------+----+-----+-----------+------+
|Customer_ID|Customer_Name|Segment|Age|Country|City|State|Postal_Code|Region|
+-----------+-------------+-------+---+-------+----+-----+-----------+------+
|          0|            0|      0|  0|      0|   0|    0|          0|     0|
+-----------+-------------+-------+---+-------+----+-----+-----------+------+



In [62]:
df = (
    Sales_df.join(Products_df, Sales_df.Product_ID == Products_df.Product_ID, "inner")
            .join(Customers_df, Sales_df.Customer_ID == Customers_df.Customer_ID, "inner")
)


In [65]:
df.printSchema()
df.show(5, truncate=False)


root
 |-- Order_Line: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)

+----------+-----------

In [74]:
from pyspark.sql.functions import desc

# Give aliases
sales = Sales_df.alias("s")
products = Products_df.alias("p")
customers = Customers_df.alias("c")

# Join
df = sales.join(products, sales.Product_ID == products.Product_ID, "inner") \
          .join(customers, sales.Customer_ID == customers.Customer_ID, "inner")




In [66]:
# 1. Total sales for each product category
q1 = df.groupBy("Category").agg(sum("Sales").alias("Total_Sales"))
q1.show()

+---------------+-----------------+
|       Category|      Total_Sales|
+---------------+-----------------+
|Office Supplies|719047.0320000029|
|      Furniture|741999.7952999998|
|     Technology|836154.0329999966|
+---------------+-----------------+



In [76]:
# 2. Customer with highest number of purchases
q2 = df.groupBy(sales.Customer_ID).count().orderBy(desc("count")).limit(1)
q2.show()

+-----------+-----+
|Customer_ID|count|
+-----------+-----+
|   WB-21850|   37|
+-----------+-----+



In [79]:
# 3. Average discount given across all products
q3 = df.agg(avg("Discount").alias("Avg_Discount"))
q3.show()

+-------------------+
|       Avg_Discount|
+-------------------+
|0.15620272163298934|
+-------------------+



In [84]:
# 4. Unique products sold in each region
q4 = df.groupBy(customers.Region).agg(countDistinct(sales.Product_ID).alias("Unique_Products"))
q4.show()

+-------+---------------+
| Region|Unique_Products|
+-------+---------------+
|  South|           1059|
|Central|           1316|
|   East|           1408|
|   West|           1536|
+-------+---------------+



In [85]:
# 5. Total profit generated in each state
q5 = df.groupBy("State").agg(sum("Profit").alias("Total_Profit"))
q5.show()


+--------------------+------------------+
|               State|      Total_Profit|
+--------------------+------------------+
|                Utah|1818.1938000000005|
|           Minnesota| 7202.522500000001|
|                Ohio| 5985.887000000001|
|              Oregon|234.04950000000014|
|            Arkansas|          -62.9462|
|               Texas| 20528.91100000002|
|        Pennsylvania|13604.935000000007|
|         Connecticut| 533.4986999999999|
|            Nebraska|         1166.0176|
|              Nevada|278.06780000000003|
|          Washington|24405.796599999983|
|            Illinois| 9560.145599999993|
|            Oklahoma| 829.0181999999999|
|District of Columbia|490.95669999999996|
|            Delaware| 3336.382700000002|
|          New Mexico|         1340.1399|
|            Missouri|2212.8746999999994|
|        Rhode Island|         2276.7013|
|             Georgia|12781.342599999998|
|            Michigan|7752.2969000000085|
+--------------------+------------

In [86]:
# 6. Product sub-category with highest sales
q6 = df.groupBy("Sub_Category").agg(sum("Sales").alias("Total_Sales")) \
       .orderBy(desc("Total_Sales")).limit(1)
q6.show()

+------------+-----------------+
|Sub_Category|      Total_Sales|
+------------+-----------------+
|      Phones|330007.0540000001|
+------------+-----------------+



In [87]:
# 7. Average age of customers in each segment
q7 = df.groupBy("Segment").agg(avg("Age").alias("Avg_Age"))
q7.show()


+-----------+------------------+
|    Segment|           Avg_Age|
+-----------+------------------+
|   Consumer| 44.60585628973223|
|Home Office| 43.28210880538418|
|  Corporate|44.816556291390725|
+-----------+------------------+



In [88]:
# 8. Orders shipped in each shipping mode
q8 = df.groupBy("Ship_Mode").count()
q8.show()

+--------------+-----+
|     Ship_Mode|count|
+--------------+-----+
|   First Class| 1538|
|      Same Day|  543|
|  Second Class| 1945|
|Standard Class| 5968|
+--------------+-----+



In [89]:
# 9. Total quantity of products sold in each city
q9 = df.groupBy("City").agg(sum("Quantity").alias("Total_Quantity"))
q9.show()

+---------------+--------------+
|           City|Total_Quantity|
+---------------+--------------+
|          Tyler|            22|
|    Springfield|           282|
|  Bowling Green|            89|
|         Auburn|           214|
|North Las Vegas|            29|
|        Phoenix|           256|
|  Lake Elsinore|            35|
|         Monroe|           184|
| Pembroke Pines|            48|
|       Westland|            54|
|    Lindenhurst|            42|
|         Marion|            77|
|          Omaha|            43|
|   Fort Collins|            55|
|        Everett|            24|
|     Greensboro|            51|
|       Franklin|           184|
|   Lincoln Park|            28|
|         Dallas|           602|
|    Thomasville|            40|
+---------------+--------------+
only showing top 20 rows



In [92]:
# 10. Customer segment with highest profit margin
df = df.withColumn("Profit_Margin", col("Profit") / col("Sales"))
q10 = df.groupBy("Segment").agg(avg("Profit_Margin").alias("Avg_Profit_Margin")) \
       .orderBy(desc("Avg_Profit_Margin")).limit(1)
q10.show()

+-----------+-------------------+
|    Segment|  Avg_Profit_Margin|
+-----------+-------------------+
|Home Office|0.14286958506103364|
+-----------+-------------------+

